# Task 1: FT-Transformer 64D vs 70D — fixed gene-disjoint 8:1:1

**目标**：在 Task 16 完全相同的 gene-disjoint 8:1:1 split 上，比较两层 FT-Transformer 使用 64D 与 70D features 的预测表现。

FT-Transformer为每个连续feature学习独立tokenisation，再通过2层self-attention建模feature interactions；它不是把70个scalar当作普通sequence直接输入。

- 64D = fold-fitted ESM2 PCA(50) + structure(7) + stability(4) + TMD(3)
- 70D = 64D + DeepLoc WT sorting-signal probabilities(6)
- validation AUPRC用于architecture selection和early stopping；test不参与选择
- 每个配置运行5个random seeds，最终比较seed-ensemble test predictions


In [1]:
from pathlib import Path
import copy
import random
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

ROOT = Path("/mnt/volume6/czj/labLGN/LabLZ")
BASE = ROOT / "xgboost_trial"
TRIAL = ROOT / "ft_trial"
TRIAL.mkdir(parents=True, exist_ok=True)
FEATURES_CSV = BASE / "features_v3.csv"
TMD_CSV = BASE / "tmd_features.csv"
CONTEXT_CSV = BASE / "deeploc_wt_context_features.csv"
SPLIT_CSV = BASE / "task16_holdout_split.csv"
PREDICTIONS_CSV = TRIAL / "task1_ft_transformer_predictions.csv"
METRICS_CSV = TRIAL / "task1_ft_transformer_metrics.csv"
SEED_METRICS_CSV = TRIAL / "task1_ft_transformer_seed_metrics.csv"
HISTORY_CSV = TRIAL / "task1_ft_transformer_training_history.csv"

RANDOM_SEEDS = [11, 22, 33, 44, 55]
N_COMPONENTS = 50
BATCH_SIZE = 32
MAX_EPOCHS = 300
PATIENCE = 30
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-3
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() and torch.cuda.device_count() > 1 else ("cuda" if torch.cuda.is_available() else "cpu"))

STRUCT_COLS = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand", "ss_coil", "delta_hydrophobicity"]
DDG_COLS = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]
TMD_COLS = ["in_TMD", "dist_to_nearest_TMD", "delta_membrane_insertion"]
WT_SIGNAL_COLS = ["deeploc_wt_signal_signal_peptide", "deeploc_wt_signal_mitochondrial_transit_peptide", "deeploc_wt_signal_nuclear_localisation_signal", "deeploc_wt_signal_nuclear_export_signal", "deeploc_wt_signal_peroxisomal_targeting_signal", "deeploc_wt_signal_gpi_anchor"]
FT_CONFIGS = {
    "ft_small": {"d_token": 32, "n_heads": 4, "d_ffn": 64, "dropout": 0.25},
    "ft_medium": {"d_token": 64, "n_heads": 4, "d_ffn": 128, "dropout": 0.30},
}
print(f"Device: {DEVICE}")

Device: cuda:1


## 1.1 Load features and reuse the Task 16 split

所有表格按canonical `KEY`做one-to-one merge。split不重新生成。

In [2]:
df = pd.read_csv(FEATURES_CSV)
assert len(df) == 2179 and df["KEY"].is_unique
assert df["Mislocalized"].isin([0, 1]).all() and int(df["Mislocalized"].sum()) == 236
for name in DDG_COLS:
    table = pd.read_csv(BASE / f"{name}.csv")
    assert table["KEY"].is_unique and name in table.columns
    df = df.merge(table[["KEY", name]], on="KEY", how="left", validate="one_to_one")
tmd = pd.read_csv(TMD_CSV)
assert tmd["KEY"].is_unique and all(c in tmd.columns for c in TMD_COLS)
df = df.merge(tmd[["KEY"] + TMD_COLS], on="KEY", how="left", validate="one_to_one")
context = pd.read_csv(CONTEXT_CSV)
assert context["KEY"].is_unique and all(c in context.columns for c in WT_SIGNAL_COLS)
df = df.merge(context[["KEY"] + WT_SIGNAL_COLS], on="KEY", how="left", validate="one_to_one")
split = pd.read_csv(SPLIT_CSV)
assert split["KEY"].is_unique and set(split["split"]) == {"train", "validation", "test"}
df = df.merge(split[["KEY", "split"]], on="KEY", how="left", validate="one_to_one")
esm_cols = [c for c in df.columns if c.startswith("esm_")]
assert len(esm_cols) == 1280 and all(c in df.columns for c in STRUCT_COLS + DDG_COLS + TMD_COLS + WT_SIGNAL_COLS)
y = df["Mislocalized"].astype(int).to_numpy()
split_indices = {name: np.flatnonzero(df["split"].to_numpy() == name) for name in ["train", "validation", "test"]}
gene_sets = {name: set(df.loc[idx, "Gene"].astype(str)) for name, idx in split_indices.items()}
assert gene_sets["train"].isdisjoint(gene_sets["validation"])
assert gene_sets["train"].isdisjoint(gene_sets["test"])
assert gene_sets["validation"].isdisjoint(gene_sets["test"])
for name, idx in split_indices.items():
    print(f"{name:10s}: n={len(idx):4d}, genes={len(gene_sets[name]):3d}, positives={int(y[idx].sum()):3d}, prevalence={y[idx].mean():.4f}")

train     : n=1748, genes=696, positives=189, prevalence=0.1081
validation: n= 221, genes= 87, positives= 24, prevalence=0.1086
test      : n= 210, genes= 88, positives= 23, prevalence=0.1095


## 1.2 Train-only preprocessing

与MLP notebook使用相同的数据矩阵。所有imputation、scaling和PCA只在train set拟合。

In [3]:
X_esm = df[esm_cols].to_numpy(np.float32)
X_struct = df[STRUCT_COLS].to_numpy(np.float32)
X_extra64 = df[DDG_COLS + TMD_COLS].to_numpy(np.float32)
X_signal = df[WT_SIGNAL_COLS].to_numpy(np.float32)
train_idx, val_idx, test_idx = split_indices["train"], split_indices["validation"], split_indices["test"]

def fit_transform_block(raw, indices):
    imputer, scaler = SimpleImputer(strategy="median"), StandardScaler()
    train = scaler.fit_transform(imputer.fit_transform(raw[indices["train"]]))
    val = scaler.transform(imputer.transform(raw[indices["validation"]]))
    test = scaler.transform(imputer.transform(raw[indices["test"]]))
    return train, val, test

esm_train, esm_val, esm_test = fit_transform_block(X_esm, split_indices)
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pc_train, pc_val, pc_test = pca.fit_transform(esm_train), pca.transform(esm_val), pca.transform(esm_test)
struct_train, struct_val, struct_test = fit_transform_block(X_struct, split_indices)
extra_train, extra_val, extra_test = fit_transform_block(X_extra64, split_indices)
signal_train, signal_val, signal_test = fit_transform_block(X_signal, split_indices)
matrices = {
    64: (np.hstack([pc_train, struct_train, extra_train]), np.hstack([pc_val, struct_val, extra_val]), np.hstack([pc_test, struct_test, extra_test])),
    70: (np.hstack([pc_train, struct_train, extra_train, signal_train]), np.hstack([pc_val, struct_val, extra_val, signal_val]), np.hstack([pc_test, struct_test, extra_test, signal_test])),
}
for dim, parts in matrices.items():
    matrices[dim] = tuple(np.asarray(x, dtype=np.float32) for x in parts)
    assert all(x.shape[1] == dim and np.isfinite(x).all() for x in matrices[dim])
    print(f"{dim}D: train={matrices[dim][0].shape}, validation={matrices[dim][1].shape}, test={matrices[dim][2].shape}")
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

64D: train=(1748, 64), validation=(221, 64), test=(210, 64)
70D: train=(1748, 70), validation=(221, 70), test=(210, 70)


## 1.3 Two-layer FT-Transformer

每个continuous feature有独立的learnable scale和bias，将scalar转换为feature token；加入CLS token后经过2层Transformer encoder。

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class NumericalFeatureTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias = nn.Parameter(torch.empty(n_features, d_token))
        nn.init.normal_(self.weight, std=0.02)
        nn.init.normal_(self.bias, std=0.02)
    def forward(self, x):
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class FTTransformer(nn.Module):
    def __init__(self, n_features, d_token, n_heads, d_ffn, dropout):
        super().__init__()
        self.tokenizer = NumericalFeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        layer = nn.TransformerEncoderLayer(d_model=d_token, nhead=n_heads, dim_feedforward=d_ffn, dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.Linear(d_token, 1))
        nn.init.normal_(self.cls_token, std=0.02)
    def forward(self, x):
        tokens = self.tokenizer(x)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        encoded = self.encoder(torch.cat([cls, tokens], dim=1))
        return self.head(encoded[:, 0]).squeeze(1)

def predict_probabilities(model, X):
    model.eval()
    with torch.inference_mode():
        logits = model(torch.as_tensor(X, dtype=torch.float32, device=DEVICE))
        return torch.sigmoid(logits).cpu().numpy()

def train_one_seed(X_train, X_val, X_test, config, seed, feature_dim, config_name):
    set_seed(seed)
    model = FTTransformer(feature_dim, **config).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    pos_weight = torch.tensor([(len(y_train) - y_train.sum()) / y_train.sum()], dtype=torch.float32, device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    dataset = TensorDataset(torch.as_tensor(X_train), torch.as_tensor(y_train, dtype=torch.float32))
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    best_ap, best_epoch, best_state, wait = -np.inf, -1, None, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            losses.append(loss.item())
        val_prob = predict_probabilities(model, X_val)
        val_ap = average_precision_score(y_val, val_prob)
        history.append({"feature_dim": feature_dim, "config": config_name, "seed": seed, "epoch": epoch, "train_loss": np.mean(losses), "validation_auprc": val_ap})
        if val_ap > best_ap + 1e-5:
            best_ap, best_epoch = val_ap, epoch
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
            wait = 0
        else:
            wait += 1
        if wait >= PATIENCE:
            break
    model.load_state_dict(best_state)
    val_prob = predict_probabilities(model, X_val)
    test_prob = predict_probabilities(model, X_test)
    record = {"feature_dim": feature_dim, "config": config_name, "seed": seed, "parameters": sum(p.numel() for p in model.parameters()), "best_epoch": best_epoch, "validation_auroc": roc_auc_score(y_val, val_prob), "validation_auprc": average_precision_score(y_val, val_prob)}
    return record, val_prob, test_prob, history


## 1.4 Configuration selection by validation AUPRC

`ft_small`与`ft_medium`都固定为2层，仅改变token width和FFN容量。分别为64D/70D选择mean validation AUPRC最高的配置。

In [5]:
seed_records, histories, prediction_store = [], [], {}
for feature_dim, (X_train_d, X_val_d, X_test_d) in matrices.items():
    for config_name, config in FT_CONFIGS.items():
        for seed in RANDOM_SEEDS:
            record, val_prob, test_prob, history = train_one_seed(X_train_d, X_val_d, X_test_d, config, seed, feature_dim, config_name)
            seed_records.append(record)
            histories.extend(history)
            prediction_store[(feature_dim, config_name, seed)] = {"validation": val_prob, "test": test_prob}
            print(f"{feature_dim}D {config_name} seed={seed}: params={record['parameters']:,}, best_epoch={record['best_epoch']}, val AUPRC={record['validation_auprc']:.4f}")
seed_metrics = pd.DataFrame(seed_records)
selection = seed_metrics.groupby(["feature_dim", "config"], as_index=False).agg(mean_validation_auprc=("validation_auprc", "mean"), sd_validation_auprc=("validation_auprc", "std"), mean_validation_auroc=("validation_auroc", "mean"))
selected = selection.sort_values(["feature_dim", "mean_validation_auprc"], ascending=[True, False]).groupby("feature_dim", as_index=False).first()
print("\nConfiguration summary:")
print(selection.to_string(index=False))
print("\nSelected without using test labels:")
print(selected.to_string(index=False))

64D ft_small seed=11: params=21,313, best_epoch=3, val AUPRC=0.2869
64D ft_small seed=22: params=21,313, best_epoch=5, val AUPRC=0.2979
64D ft_small seed=33: params=21,313, best_epoch=5, val AUPRC=0.2399
64D ft_small seed=44: params=21,313, best_epoch=3, val AUPRC=0.2124
64D ft_small seed=55: params=21,313, best_epoch=21, val AUPRC=0.2817
64D ft_medium seed=11: params=75,393, best_epoch=14, val AUPRC=0.2498
64D ft_medium seed=22: params=75,393, best_epoch=3, val AUPRC=0.2633
64D ft_medium seed=33: params=75,393, best_epoch=18, val AUPRC=0.2789
64D ft_medium seed=44: params=75,393, best_epoch=8, val AUPRC=0.2508
64D ft_medium seed=55: params=75,393, best_epoch=2, val AUPRC=0.2875
70D ft_small seed=11: params=21,697, best_epoch=3, val AUPRC=0.2225
70D ft_small seed=22: params=21,697, best_epoch=2, val AUPRC=0.2792
70D ft_small seed=33: params=21,697, best_epoch=24, val AUPRC=0.2712
70D ft_small seed=44: params=21,697, best_epoch=5, val AUPRC=0.3056
70D ft_small seed=55: params=21,697, be

## 1.5 Final seed-ensemble test evaluation

对selected configuration的5个seed probabilities取平均，再计算test AUROC/AUPRC。

In [6]:
metric_rows, output = [], df[["KEY", "Gene", "Mutation_used", "Mislocalized", "split"]].copy()
for row in selected.itertuples(index=False):
    feature_dim, config_name = int(row.feature_dim), row.config
    val_probs = np.vstack([prediction_store[(feature_dim, config_name, seed)]["validation"] for seed in RANDOM_SEEDS])
    test_probs = np.vstack([prediction_store[(feature_dim, config_name, seed)]["test"] for seed in RANDOM_SEEDS])
    val_ensemble, test_ensemble = val_probs.mean(axis=0), test_probs.mean(axis=0)
    metric_rows.extend([
        {"scope": "validation_seed_ensemble", "feature_dim": feature_dim, "config": config_name, "n": len(y_val), "positives": int(y_val.sum()), "auroc": roc_auc_score(y_val, val_ensemble), "auprc": average_precision_score(y_val, val_ensemble), "brier": brier_score_loss(y_val, val_ensemble)},
        {"scope": "test_seed_ensemble", "feature_dim": feature_dim, "config": config_name, "n": len(y_test), "positives": int(y_test.sum()), "auroc": roc_auc_score(y_test, test_ensemble), "auprc": average_precision_score(y_test, test_ensemble), "brier": brier_score_loss(y_test, test_ensemble)},
    ])
    output[f"ft_transformer_{feature_dim}d_probability"] = np.nan
    output.loc[val_idx, f"ft_transformer_{feature_dim}d_probability"] = val_ensemble
    output.loc[test_idx, f"ft_transformer_{feature_dim}d_probability"] = test_ensemble
    for seed in RANDOM_SEEDS:
        prob = prediction_store[(feature_dim, config_name, seed)]["test"]
        mask = (seed_metrics["feature_dim"] == feature_dim) & (seed_metrics["config"] == config_name) & (seed_metrics["seed"] == seed)
        seed_metrics.loc[mask, "test_auroc"] = roc_auc_score(y_test, prob)
        seed_metrics.loc[mask, "test_auprc"] = average_precision_score(y_test, prob)
metrics = pd.DataFrame(metric_rows)
print(metrics.to_string(index=False))
output.to_csv(PREDICTIONS_CSV, index=False)
metrics.to_csv(METRICS_CSV, index=False)
seed_metrics.to_csv(SEED_METRICS_CSV, index=False)
pd.DataFrame(histories).to_csv(HISTORY_CSV, index=False)
print(f"Saved: {PREDICTIONS_CSV}")
print(f"Saved: {METRICS_CSV}")

                   scope  feature_dim    config   n  positives    auroc    auprc    brier
validation_seed_ensemble           64 ft_medium 221         24 0.682530 0.277097 0.159551
      test_seed_ensemble           64 ft_medium 210         23 0.581958 0.194272 0.189014
validation_seed_ensemble           70 ft_medium 221         24 0.613367 0.262197 0.147016
      test_seed_ensemble           70 ft_medium 210         23 0.540572 0.212997 0.154945
Saved: /mnt/volume6/czj/labLGN/LabLZ/ft_trial/task1_ft_transformer_predictions.csv
Saved: /mnt/volume6/czj/labLGN/LabLZ/ft_trial/task1_ft_transformer_metrics.csv


## Interpretation boundary

这是约23个positive的单次held-out test，只适合快速筛选。若FT-Transformer显示稳定优势，应继续完成five-fold gene-disjoint OOF和paired gene-cluster bootstrap。看到test结果后继续调参会使该test失去独立性。